# Dynamic Advertisement Selection Using Multi-Armed Bandit & Contextual Reinforcement Learning

**Authors:** Reinforcement Learning Capstone Team  
**Domain:** Online Advertising / Reinforcement Learning / Multi-Armed Bandits  
**Notebook Scope:** Complete End-to-End Experimental Benchmark from Classical UCB1 to Contextual Personalization & Non-Stationary Drift.

## 1. Setup & Environment Imports
We import our modular from-scratch packages from `src/`.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.environment import AdvertisementEnvironment
from src.ucb_agent import UCB1Agent
from src.baseline_agents import RandomAgent, EpsilonGreedyAgent, ThompsonSamplingAgent
from src.contextual_environment import ContextualAdvertisementEnvironment
from src.contextual_agent import LinUCBAgent
from src.non_stationary import NonStationaryEnvironment, DiscountedUCBAgent
from src.offline_evaluator import generate_sample_ad_dataset, evaluate_policy_offline_replay
from src.simulation import (
    run_simulation,
    run_multi_agent_simulation,
    run_contextual_simulation,
    run_non_stationary_simulation,
)
from src.evaluation import calculate_metrics, generate_plots, generate_multi_agent_plots

print('All modules imported successfully!')

## 2. Experiment 1: Single-Agent UCB1 Benchmark
We formulate 5 advertisements with hidden Click-Through Rates (CTRs): `[0.05, 0.10, 0.03, 0.15, 0.08]`.
The optimal arm is Ad 4 with CTR = 0.15.

In [ ]:
true_ctrs = [0.05, 0.10, 0.03, 0.15, 0.08]
ad_names = ['Ad 1', 'Ad 2', 'Ad 3', 'Ad 4', 'Ad 5']
env = AdvertisementEnvironment(true_ctrs=true_ctrs, ad_names=ad_names, seed=42)
agent = UCB1Agent(num_ads=5, c=2.0)

df_ucb = run_simulation(env, agent, num_rounds=10000, seed=42)
metrics_ucb = calculate_metrics(df_ucb, env, agent)

print(f"Total Clicks Harvested: {metrics_ucb['total_reward']}")
print(f"Average System CTR: {metrics_ucb['average_reward']:.4f}")
print(f"Most Selected Ad: {metrics_ucb['agent_favorite_name']}")
metrics_ucb['ad_summary_df']

## 3. Experiment 2: Multi-Algorithm Benchmark Arena
We benchmark UCB1 against:
1. **Random / Static A/B Testing** (Uniform Exploration)
2. **ε-Greedy** (Fixed ε=0.1)
3. **ε-Greedy with Decay** (ε_t = min(1.0, 5 / t))
4. **Thompson Sampling** (Bayesian Beta-Bernoulli Posterior Sampling)

In [ ]:
agents = {
    'UCB1': UCB1Agent(num_ads=5, c=2.0),
    'Thompson Sampling': ThompsonSamplingAgent(num_ads=5, seed=42),
    'ε-Greedy (decay)': EpsilonGreedyAgent(num_ads=5, epsilon=0.5, decay=True, seed=42),
    'ε-Greedy (ε=0.1)': EpsilonGreedyAgent(num_ads=5, epsilon=0.1, decay=False, seed=42),
    'Random / Static': RandomAgent(num_ads=5, seed=42),
}

multi_results = run_multi_agent_simulation(agents=agents, true_ctrs=true_ctrs, num_rounds=10000, seed=42)

leaderboard = []
for name, df in multi_results.items():
    clicks = int(df['cumulative_reward'].iloc[-1])
    regret = float(df['cumulative_expected_regret'].iloc[-1])
    ctr = clicks / len(df)
    leaderboard.append({
        'Algorithm': name,
        'Total Clicks': clicks,
        'Effective CTR': f'{ctr:.4f}',
        'Cumulative Regret': f'{regret:.2f}',
    })

pd.DataFrame(leaderboard).sort_values(by='Total Clicks', ascending=False)

## 4. Visualizing Regret & Harvested Reward Trajectories

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 5))

for name, df in multi_results.items():
    axs[0].plot(df['round'], df['cumulative_expected_regret'], label=name, linewidth=2)
    axs[1].plot(df['round'], df['cumulative_reward'], label=name, linewidth=2)

axs[0].set_title('Cumulative Regret (Sublinear vs Linear Growth)', fontweight='bold')
axs[0].set_xlabel('Round (t)')
axs[0].set_ylabel('Cumulative Expected Regret')
axs[0].grid(True, linestyle='--', alpha=0.6)
axs[0].legend()

axs[1].set_title('Cumulative Clicks Harvested Over Time', fontweight='bold')
axs[1].set_xlabel('Round (t)')
axs[1].set_ylabel('Clicks')
axs[1].grid(True, linestyle='--', alpha=0.6)
axs[1].legend()

plt.tight_layout()
plt.show()

## 5. Experiment 3: Contextual Bandits (Disjoint LinUCB Personalization)
In real-world advertising, CTR is conditioned on user context vectors x_t in R^d.
LinUCB uses ridge regression to estimate weight vectors theta_hat_a for each ad arm.

In [ ]:
c_env = ContextualAdvertisementEnvironment(num_ads=4, context_dim=5, seed=42)
c_agent = LinUCBAgent(num_ads=4, context_dim=5, alpha=1.0)

df_contextual = run_contextual_simulation(c_env, c_agent, num_rounds=10000, seed=42)

print(f"LinUCB Total Clicks: {df_contextual['cumulative_reward'].iloc[-1]}")
print(f"LinUCB Cumulative Contextual Regret: {df_contextual['cumulative_regret'].iloc[-1]:.2f}")

learned_thetas = c_agent.get_estimated_thetas()
print('Learned LinUCB Theta Matrix:')
print(np.round(learned_thetas, 3))
print('\nGround Truth Theta Matrix:')
print(np.round(c_env.theta_matrix, 3))

## 6. Experiment 4: Non-Stationary Dynamics & Ad Fatigue
We simulate a regime shift at round t=5000: Ad 4 CTR collapses (0.15 -> 0.02) due to audience ad fatigue, while Ad 2 surges (0.10 -> 0.18).
We compare Stationary UCB1 against Discounted-UCB (D-UCB) with discount factor gamma=0.995.

In [ ]:
ns_env = NonStationaryEnvironment(
    initial_ctrs=[0.05, 0.10, 0.03, 0.15, 0.08],
    change_points={5000: [0.05, 0.18, 0.03, 0.02, 0.08]},
    seed=42,
)

ns_agents = {
    'Stationary UCB1': UCB1Agent(num_ads=5, c=1.5),
    'Discounted-UCB (γ=0.995)': DiscountedUCBAgent(num_ads=5, gamma=0.995, c=1.0),
}

ns_results = run_non_stationary_simulation(ns_env, ns_agents, num_rounds=10000, seed=42)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ns_results['Stationary UCB1']['round'], ns_results['Stationary UCB1']['cumulative_regret'], label='Stationary UCB1', color='#dc2626', linewidth=2.2)
ax.plot(ns_results['Discounted-UCB (γ=0.995)']['round'], ns_results['Discounted-UCB (γ=0.995)']['cumulative_regret'], label='Discounted-UCB', color='#16a34a', linewidth=2.2)
ax.axvline(x=5000, color='black', linestyle=':', linewidth=2, label='Regime Shift (t=5000)')
ax.set_title('Non-Stationary Ad Fatigue Adaptation: UCB1 vs Discounted-UCB', fontweight='bold')
ax.set_xlabel('Simulation Round (t)')
ax.set_ylabel('Cumulative Regret')
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend()
plt.tight_layout()
plt.show()

## 7. Experiment 5: Unbiased Offline Policy Replay Evaluation
We evaluate the policies on offline logged click data using the Li et al. (2010) Replay Method.

In [ ]:
df_logs = pd.read_csv('../data/real_ad_clicks.csv')
print(f'Loaded offline log with {len(df_logs)} records.')

eval_ucb = evaluate_policy_offline_replay(UCB1Agent(num_ads=4, c=1.5), df_logs, is_contextual=False)
eval_ts = evaluate_policy_offline_replay(ThompsonSamplingAgent(num_ads=4, seed=42), df_logs, is_contextual=False)
eval_linucb = evaluate_policy_offline_replay(LinUCBAgent(num_ads=4, context_dim=5, alpha=1.0), df_logs, is_contextual=True)

offline_summary = pd.DataFrame([
    {'Algorithm': 'UCB1', 'Accepted Impressions': eval_ucb['accepted_rounds'], 'Total Clicks': eval_ucb['total_clicks'], 'Replay Effective CTR': f"{eval_ucb['effective_ctr']:.4f}"},
    {'Algorithm': 'Thompson Sampling', 'Accepted Impressions': eval_ts['accepted_rounds'], 'Total Clicks': eval_ts['total_clicks'], 'Replay Effective CTR': f"{eval_ts['effective_ctr']:.4f}"},
    {'Algorithm': 'Contextual LinUCB', 'Accepted Impressions': eval_linucb['accepted_rounds'], 'Total Clicks': eval_linucb['total_clicks'], 'Replay Effective CTR': f"{eval_linucb['effective_ctr']:.4f}"},
])
offline_summary

## 8. Conclusion & Capstone Key Takeaways
1. **Logarithmic Regret Guaranteed:** Both UCB1 and Thompson Sampling exhibit O(ln T) sublinear regret, vastly outperforming static A/B testing and fixed ε-greedy (O(T)).
2. **Thompson Sampling Performance:** Thompson Sampling achieves the lowest empirical regret in stationary Bernoulli settings.
3. **Personalization Power of LinUCB:** Contextual bandits boost CTR by matching demographic preferences to tailored advertisement arms.
4. **Non-Stationary Resilience:** Discounted-UCB dynamically forgets obsolete history and adapts immediately when ad fatigue occurs.